In [ ]:
# Enrichment analysis that report log enrichments scores with replicate uncertainty as volcano plot analyses 
'''
Enrichment of synthetic combinatorial peptide libraries 

1. **Peptide Finder** – filter de novo peptide IDs against the library design
   (length, PTM pattern, fixed residues, allowed residues at variable positions, ALC cut-off).
2. **Enrichment score** per peptide and run: `Area/mean(Area) + Count/mean(Count)`.
3. **Replicate merging** – relative enrichment (sample vs. control) + t-test p-values.
4. **Visualisation** – scatter, volcano plots, sequence logos, amino-acid heatmaps.

Input: one sub-folder per LC-MS run inside `base_dir`, each containing `merged_peaks_output.csv`.
See `README.txt` for details.
'''

In [ ]:
# Import packages and set paths

from pathlib import Path
import csv
import re  # Regular expression library
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

# -----------------------------
# Configuration for paths
# -----------------------------

# Base directory: contains one sub-folder per LC-MS run (replicate)
# >>> EDIT THIS to point to your data folder <<<
base_dir = Path(r'path')

# Name of the de novo export inside each run folder
input_name = 'merged_peaks_output.csv' # this file contains "all de novo peptides" from PEAKS Studio sequencing including the MS1-based area of sequenced peptides


# Name of the filtered peptide table written by the Peptide Finder step
filtered_name = 'filtered-peptides-allowedletters.csv'

# Map condition names to folder-name patterns; folders are '<prefix>-1' ... '<prefix>-3'
conditions = {
    'Control': 'supernatant',
    'Sample':  'pellet',
}

n_replicates = 3

print(f"Base directory: {base_dir.resolve()}")

In [ ]:
# ----------------------------------------------------------------
# Filter settings (Peptide Finder) — library design
# ----------------------------------------------------------------

# Library FL24
length_criteria = 5
ptm_positions = [0, 4]
specific_letters = ['K']
specific_positions = [4]
alc_threshold = 0

# Allowed letters and their positions
allowed_letters = {'W', 'R', 'N', 'D'}  # letters used in variable positions
allowed_positions = {0, 1, 2, 3}        # positions of variable letters (0-indexed)

# Amino acids to track (used in the heatmaps)
amino_acids_to_track = ['W', 'R', 'N', 'D', 'K']

In [4]:
# Precompile regex once
AA_PTM = re.compile(r'([A-Z])((?:\(\+?[\d\.-]+\))+)?')  # AA followed by one or more PTMs


def process_peptide(peptide):
    """
    Parse peptide 'A(+42.01)B...' into list of dicts:
    {'letter': 'A', 'PTMs': True/False, 'position': i}
    (PTMs as boolean to avoid object churn)
    """
    result = []
    for index, (amino_acid, ptms) in enumerate(AA_PTM.findall(peptide)):
        result.append({'letter': amino_acid, 'PTMs': bool(ptms), 'position': index})
    return result


def check_specific_letters_positions(processed_peptide, allowed_letters_set, allowed_positions_set):
    """Same allowed_letters set for all allowed (variable) positions."""
    for pos in allowed_positions_set:
        if pos >= len(processed_peptide):
            return False
        if processed_peptide[pos]['letter'] not in allowed_letters_set:
            return False
    return True


def filter_peptides(
    input_file, output_file, length_criteria, ptm_positions,
    specific_letters=None, specific_positions=None,
    allowed_letters=None, allowed_positions=None
):
    """Write all rows of input_file that match the library design to output_file."""
    # Precompute sets
    ptm_positions_set = set(ptm_positions) if ptm_positions else None
    allowed_letters_set = set(allowed_letters) if allowed_letters else None
    allowed_positions_set = set(allowed_positions) if allowed_positions else None

    with open(input_file, mode='r', newline='') as infile:
        reader = csv.DictReader(infile)
        fieldnames = reader.fieldnames

        with open(output_file, mode='w', newline='') as outfile:
            writer = csv.DictWriter(outfile, fieldnames=fieldnames)
            writer.writeheader()

            for row in reader:
                # Cheap check first
                if alc_threshold is not None:
                    try:
                        if float(row['ALC (%)']) < alc_threshold:
                            continue
                    except (KeyError, ValueError):
                        continue

                processed_peptide = process_peptide(row['Peptide'])

                # Length
                if len(processed_peptide) != length_criteria:
                    continue

                # PTM positions (exact set match)
                if ptm_positions_set is not None:
                    ptm_positions_found = {aa['position'] for aa in processed_peptide if aa['PTMs']}
                    if ptm_positions_found != ptm_positions_set:
                        continue

                # Specific fixed letters
                if specific_letters and specific_positions:
                    if not all(
                        processed_peptide[pos]['letter'] == letter
                        for letter, pos in zip(specific_letters, specific_positions)
                    ):
                        continue

                # Allowed letters at the variable positions
                if allowed_letters_set and allowed_positions_set:
                    if not check_specific_letters_positions(processed_peptide,
                                                            allowed_letters_set,
                                                            allowed_positions_set):
                        continue

                # Keep duplicates (no dedup) — duplicates are counted in the scoring step
                writer.writerow(row)

In [ ]:
# ------------------------------
# 1) Run "Peptide Finder" filter with one allowed-letter set (serial)

jobs = []
for subdir in base_dir.iterdir():
    if not subdir.is_dir():
        continue
    inp = subdir / input_name
    outp = subdir / filtered_name
    if inp.exists():
        jobs.append((inp, outp))

print(f"Found {len(jobs)} inputs.")

for inp, outp in jobs:
    try:
        filter_peptides(
            inp, outp,
            length_criteria, ptm_positions,
            specific_letters, specific_positions,
            allowed_letters, allowed_positions
        )
    except Exception as e:
        print(f"[ERROR] {inp}: {e}")

In [ ]:
# --------------------------------------------------
# 2) Compute Enrichment score per peptide (per sub-run)

for subdir in base_dir.iterdir():
    if subdir.is_dir():
        input_file = subdir / filtered_name
        output_file = subdir / 'peptide_enrichment_scores.csv'
        if input_file.exists():
            print(f"Processing: {input_file}")

            # Load CSV
            df = pd.read_csv(input_file)
            df['Scan'] = df['Scan'].astype(str)

            # Clean peptide names (remove PTM tags BEFORE grouping)
            df['Peptide'] = df['Peptide'].str.replace(r'\([^)]*\)', '', regex=True)

            # Create Area_for_avg that ignores zeros when averaging
            tmp = df.copy()
            tmp['Area_for_avg'] = tmp['Area'].replace(0, pd.NA)

            # Per-peptide averages (Area_for_avg, ALC)
            area_alc_df = tmp.groupby('Peptide').agg({
                'ALC (%)': 'mean',
                'Area_for_avg': 'mean'
            }).rename(columns={'Area_for_avg': 'Area'})

            # Peptide counts: use 'Source File'; fall back to 'Scan' if not present
            if 'Source File' in df.columns:
                count_df = df.groupby('Peptide').agg({'Source File': 'count'}).rename(columns={'Source File': 'Count'})
            else:
                count_df = df.groupby('Peptide').agg({'Scan': 'count'}).rename(columns={'Scan': 'Count'})

            # Merge
            grouped = area_alc_df.join(count_df)

            # Run-wide averages (computed on the per-peptide averages)
            run_avg_area = grouped['Area'].mean(skipna=True)
            run_avg_count = grouped['Count'].mean(skipna=True)

            # Enrichment score
            grouped['EnrichmentScore'] = (
                (grouped['Area'] / run_avg_area) + (grouped['Count'] / run_avg_count)
            )

            # Save to CSV
            grouped.reset_index().to_csv(output_file, index=False, float_format='%.4f', na_rep='')
            print(f"Saved: {output_file}")
        else:
            print(f"Skipped (not found): {input_file}")

In [ ]:
# ----------------------------------------------------------------
# 3) Merge replicates and compute enrichment + p-values
#    p-value threshold used downstream is 0.2 (matches the hit-selection criteria)

# In step 3 — match folders by substring, read the replicate number from the suffix
all_data = []
for condition, key in conditions.items():
    for folder in sorted(base_dir.iterdir()):
        if not folder.is_dir() or key not in folder.name:
            continue
        m = re.search(r'-(\d+)$', folder.name)     # trailing "-1", "-2", ...
        if not m:
            print(f"[WARN] no replicate number in folder name: {folder.name}")
            continue
        rep = int(m.group(1))

        file = folder / 'peptide_enrichment_scores.csv'
        if not file.exists():
            print(f"Missing: {file}")
            continue

        df = pd.read_csv(file)[['Peptide', 'EnrichmentScore']]
        df['Condition'] = condition
        df['Replicate'] = rep
        all_data.append(df)
        print(f"Loaded {condition} rep {rep}: {folder.name}")

if not all_data:
    present = [d.name for d in base_dir.iterdir() if d.is_dir()]
    raise FileNotFoundError(
        "No 'peptide_enrichment_scores.csv' loaded.\n"
        f"base_dir: {base_dir}\n"
        f"folders present: {present}\n"
        f"condition keys: {conditions}"
    )

combined_df = pd.concat(all_data, ignore_index=True)

# NOTE: rows with a missing EnrichmentScore are NOT dropped — absences are zero-filled in the pivot

# Pivot to wide and fill absences with zero (explicit 0 for peptides missing in a run)
pivot_df = combined_df.pivot_table(
    index='Peptide',
    columns=['Condition', 'Replicate'],
    values='EnrichmentScore',
    fill_value=0.0
).fillna(0.0)

# Flatten multi-index columns
pivot_df.columns = [f"{cond.capitalize()}_{rep}" for cond, rep in pivot_df.columns]


def compute_summary(row):
    # Arrays per group (always length 3 after zero-fill)
    ctrl_vals = row[[c for c in row.index if c.startswith('Control_')]].values.astype(float)
    samp_vals = row[[c for c in row.index if c.startswith('Sample_')]].values.astype(float)

    # Sums for enrichment
    sum_ctrl = float(np.sum(ctrl_vals))
    sum_samp = float(np.sum(samp_vals))

    # Enrichment with pseudocount logic
    if (sum_ctrl > 0) and (sum_samp > 0):
        rel_enrichment = sum_samp / sum_ctrl
        log2_enrichment = np.log2(rel_enrichment)
    elif (sum_ctrl == 0) and (sum_samp > 0):
        rel_enrichment = sum_samp / 0.1  # small pseudocount
        log2_enrichment = np.log2(rel_enrichment)
    elif (sum_samp == 0) and (sum_ctrl > 0):
        rel_enrichment = 0.1 / sum_ctrl  # small pseudocount
        log2_enrichment = np.log2(rel_enrichment)
    else:
        rel_enrichment = 0.0
        log2_enrichment = 0.0

    # P-value
    protein_var = (len(set(samp_vals)) > 1)
    control_var = (len(set(ctrl_vals)) > 1)

    if protein_var or control_var:
        try:
            t_stat, p_value = stats.ttest_ind(samp_vals, ctrl_vals)  # equal_var=True
            if (not np.isfinite(p_value)) or (p_value == 0):
                if np.mean(samp_vals) != np.mean(ctrl_vals):
                    p_value = 1e-10
                else:
                    p_value = 1.0
        except Exception:
            p_value = 1.0
    else:
        if np.mean(samp_vals) != np.mean(ctrl_vals):
            p_value = 0.01
        else:
            p_value = 1.0

    # Cap extremes globally to [1e-10, 1.0]
    p_value = max(min(p_value, 1.0), 1e-10)

    neg_log10_p = -np.log10(p_value) if p_value > 0 else np.nan

    return pd.Series({
        'Sum_Control': sum_ctrl,
        'Sum_Sample': sum_samp,
        'Rel_Enrichment': rel_enrichment,
        'Log2_Enrichment': log2_enrichment,
        'p-value': p_value,
        'Neg_Log10_P': neg_log10_p
    })


summary = pivot_df.apply(compute_summary, axis=1)

# Combine all into final dataframe
final_df = pivot_df.join(summary).reset_index()

# Sort (alternatively sort by Neg_Log10_P or Rel_Enrichment)
final_df = final_df.sort_values(
    by=['Sum_Sample'],
    ascending=[False]
)

# Save to CSV
output_file = base_dir / 'enrichment_score_comparison_full.csv'
final_df.to_csv(output_file, index=False, float_format='%.4g')
print(f"Saved full summary with enrichment metrics: {output_file}")

In [ ]:
# 4) Plots / visualization
# %matplotlib qt
%matplotlib inline
import matplotlib.colors as mcolors
from matplotlib.ticker import MaxNLocator

# Load the final CSV
output_file = base_dir / 'enrichment_score_comparison_full.csv'
df = pd.read_csv(output_file)

# -------------------------------
# Plot 1: Sum_Control vs Sum_Sample with Rel_Enrichment color scale (robust + adaptive)
# -------------------------------
rel_enrichment = pd.to_numeric(df['Rel_Enrichment'], errors='coerce')

mask_hi = rel_enrichment > 2
mask_lo = ~mask_hi  # includes NaN & <= 2 (all plotted grey)

plt.figure(figsize=(7, 6), dpi=120)

# 1) low/grey layer
plt.scatter(
    df.loc[mask_lo, 'Sum_Sample'],
    df.loc[mask_lo, 'Sum_Control'],
    color='lightgrey',
    edgecolor='none',
    alpha=0.6,
    label='Rel_Enrichment ≤ 2'
)

# 2) high/colored layer
vals = rel_enrichment.loc[mask_hi]
x_hi = df.loc[mask_hi, 'Sum_Sample']
y_hi = df.loc[mask_hi, 'Sum_Control']

if len(vals) > 0:
    # Adaptive vmin/vmax with a little padding
    vmin = max(2.0, float(np.nanpercentile(vals, 2)))
    vmax = float(np.nanpercentile(vals, 98))
    if not np.isfinite(vmax) or vmax <= vmin:
        vmax = float(np.nanmax(vals))

    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    sc1 = plt.scatter(
        x_hi, y_hi,
        c=vals,
        cmap='viridis',
        norm=norm,
        edgecolor='none',
        alpha=0.9,
        label='Rel_Enrichment > 2'
    )

    # Adaptive colorbar
    cb = plt.colorbar(sc1, label='Relative Enrichment (> 2)')
    cb.locator = MaxNLocator(nbins=6)
    cb.update_ticks()
else:
    # No >2 points
    sc1 = None

plt.xlabel('Sum_Sample')
plt.ylabel('Sum_Control')
plt.title('Sum_Control vs Sum_Sample\nColored by Relative Enrichment (>2 in color, ≤2 grey)')
plt.tight_layout()

scatter_fig = base_dir / 'scatter_colored_above_2.png'
plt.savefig(scatter_fig, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# -------------------------------
# Plot 2: Volcano Plot (with strict non-overlapping labels for red points)
# -------------------------------
def volcano_color(row):
    if row['p-value'] < 0.2:
        if row['Rel_Enrichment'] > 2:
            return 'red'
        elif row['Rel_Enrichment'] < 0.5:
            return 'blue'
    return 'grey'

# Colors
df['volcano_color'] = df.apply(volcano_color, axis=1)

# Significant enriched ("red") points
df_red = df[(df['p-value'] < 0.2) & (df['Rel_Enrichment'] > 2)].copy()

fig, ax = plt.subplots(figsize=(8, 6), dpi=120)

# Scatter all points
ax.scatter(
    df['Log2_Enrichment'],
    df['Neg_Log10_P'],
    c=df['volcano_color'],
    edgecolor='none',
    alpha=0.5
)

# Save red points as CSV
red_csv = base_dir / "volcano_red_points.csv"
df_red.to_csv(red_csv, index=False)
print(f"[OK] Saved {len(df_red)} red points → {red_csv}")

# --- Place non-overlapping labels for red points ---
# Sort by significance (largest -log10 p first) so the most significant get priority
df_red_sorted = df_red.sort_values('Neg_Log10_P', ascending=False)

# Each text's pixel bounding box is checked against the already-placed ones
fig.canvas.draw()  # ensure a renderer is available
renderer = fig.canvas.get_renderer()
placed_bboxes = []


def _place_label_if_no_overlap(ax, x, y, text, placed_bboxes, pad=(1.05, 1.15), fontsize=8):
    """Try to place a label; return True if placed, else False (removed)."""
    t = ax.text(x, y, text, fontsize=fontsize, ha='right', va='bottom')
    # Need a draw to get a valid bbox
    fig.canvas.draw()
    bbox = t.get_window_extent(renderer=renderer).expanded(pad[0], pad[1])
    for pb in placed_bboxes:
        if bbox.overlaps(pb):
            t.remove()
            return False
    placed_bboxes.append(bbox)
    return True


placed_count = 0
for _, row in df_red_sorted.iterrows():
    x = row['Log2_Enrichment']
    y = row['Neg_Log10_P']
    if np.isfinite(x) and np.isfinite(y):
        if _place_label_if_no_overlap(ax, x, y, str(row['Peptide']), placed_bboxes):
            placed_count += 1

print(f"[INFO] Volcano labels placed (non-overlapping): {placed_count} / {len(df_red_sorted)}")

# Threshold lines
ax.axhline(-np.log10(0.2), color='black', linestyle='--', linewidth=1)
ax.axvline(np.log2(2),     color='black', linestyle='--', linewidth=1)
ax.axvline(np.log2(0.5),   color='black', linestyle='--', linewidth=1)

ax.set_xlabel('Log2(Relative Enrichment)')
ax.set_ylabel('-Log10(p-value)')
ax.set_title('Volcano Plot (labels shown only when non-overlapping)')
plt.tight_layout()

volcano_fig = base_dir / 'volcano_with_labels.png'
plt.savefig(volcano_fig, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# -------------------------------
# Plot 2b: Volcano Plot (journal style)
# -------------------------------
def volcano_color(row):
    if row['p-value'] < 0.2:
        if row['Rel_Enrichment'] > 2:
            return '#d62728'  # soft red
        elif row['Rel_Enrichment'] < 0.5:
            return '#1f77b4'  # soft blue
    return 'lightgrey'

# Assign colors
df['volcano_color'] = df.apply(volcano_color, axis=1)

# Significant enriched ("red") points
df_red = df[(df['p-value'] < 0.2) & (df['Rel_Enrichment'] > 2)].copy()

# -------------------------------
# Figure setup (journal style)
# -------------------------------
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(3.5, 3.2), dpi=300)

# Scatter all points
ax.scatter(
    df['Log2_Enrichment'],
    df['Neg_Log10_P'],
    c=df['volcano_color'],
    s=18,
    edgecolor='none',
    alpha=0.7,
)

# Save red points as CSV
red_csv = base_dir / "volcano_red_points.csv"
df_red.to_csv(red_csv, index=False)
print(f"[OK] Saved {len(df_red)} red points → {red_csv}")

# -------------------------------
# Label non-overlapping red points (off by default for the journal figure)
# -------------------------------
label_red_points = False

df_red_sorted = df_red.sort_values('Neg_Log10_P', ascending=False)

fig.canvas.draw()
renderer = fig.canvas.get_renderer()
placed_bboxes = []

placed_count = 0
if label_red_points:
    for _, row in df_red_sorted.iterrows():
        x = row['Log2_Enrichment']
        y = row['Neg_Log10_P']
        if np.isfinite(x) and np.isfinite(y):
            if _place_label_if_no_overlap(ax, x, y, str(row['Peptide']),
                                          placed_bboxes, pad=(1.05, 1.1), fontsize=18):
                placed_count += 1

print(f"[INFO] Volcano labels placed (non-overlapping): {placed_count} / {len(df_red_sorted)}")

# -------------------------------
# Axes styling and thresholds
# -------------------------------
ax.axhline(-np.log10(0.2), color='grey', linestyle='--', linewidth=0.3)
ax.axvline(np.log2(2),     color='grey', linestyle='--', linewidth=0.3)
ax.axvline(np.log2(0.5),   color='grey', linestyle='--', linewidth=0.3)

# Axis labels (mathtext, no font warnings)
ax.set_xlabel(r'log$_2$ relative enrichment', fontsize=18)
ax.set_ylabel(r'-log$_{10}$ p-value', fontsize=18)
ax.set_xlim(df['Log2_Enrichment'].min() - 0.5, df['Log2_Enrichment'].max() + 0.5)
ax.set_ylim(bottom=0)

# No grid, so ticks align with the axis
ax.grid(False)

# Bring spines forward (ensure ticks overlap spines, not behind)
for spine in ax.spines.values():
    spine.set_zorder(10)

# Ticks cutting through the axis line
ax.tick_params(
    axis='both',
    which='major',
    direction='out',
    length=1.5,
    width=0.5,
    color='grey',
    labelsize=14,
)

# Clean spines
ax.spines['left'].set_linewidth(0.8)
ax.spines['bottom'].set_linewidth(0.8)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

plt.tight_layout(pad=1)

volcano_fig = base_dir / 'volcano_with_labels_journal.png'
plt.savefig(volcano_fig, dpi=600, bbox_inches='tight', transparent=True)
plt.show()

In [ ]:
# WebLogo of red points (df_red)

# Helper: keep only letters (strip PTMs like (+42.01))
def _clean_seq(s: str) -> str:
    return re.sub(r'[^A-Za-z]', '', str(s)).upper()


# Prepare sequences
if df_red is None or df_red.empty:
    raise ValueError("df_red is empty; nothing to plot.")

seqs = df_red['Peptide'].astype(str).map(_clean_seq)
lengths = seqs.str.len()
if lengths.empty or lengths.max() == 0:
    raise ValueError("No valid peptide sequences for logo.")

# Use the most common length to ensure alignment
L_mode = int(lengths.mode().iloc[0])
seqs_aligned = seqs[lengths == L_mode].tolist()
print(f"[INFO] WebLogo: using {len(seqs_aligned)} sequences of length {L_mode} (mode length).")

# Build counts matrix (positions x amino acids)
alphabet = list("ACDEFGHIKLMNPQRSTVWY")
counts = pd.DataFrame(0, index=range(L_mode), columns=alphabet, dtype=float)
for s in seqs_aligned:
    for i, aa in enumerate(s):
        if aa in counts.columns:
            counts.at[i, aa] += 1

# Convert to probabilities for logo heights
probs = counts.div(counts.sum(axis=1).replace(0, np.nan), axis=0)

# Try importing logomaker; give a friendly message if missing
try:
    import logomaker as lm
except ImportError as e:
    raise ImportError(
        "The 'logomaker' package is required for the WebLogo. "
        "Install it with:\n    pip install logomaker"
    ) from e

# Plot & save
plt.figure(figsize=(min(16, 0.6 * L_mode + 2), 4), dpi=300)
lm.Logo(probs.fillna(0.0), shade_below=0.5, fade_below=0.5, font_name='Arial')
plt.xlabel("Position (1-based)")
plt.ylabel("Frequency")
plt.title("WebLogo – red peptides")
plt.xticks(ticks=np.arange(L_mode), labels=np.arange(1, L_mode + 1))
plt.tight_layout()

weblogo_png = base_dir / "weblogo_red_peptides.png"
plt.savefig(weblogo_png, bbox_inches='tight', dpi=300)
plt.show()
print(f"[OK] Saved WebLogo → {weblogo_png}")

In [ ]:
# WebLogo of red points (df_red) — custom colors, journal style
from matplotlib.ticker import MaxNLocator, FixedLocator

# --- Define color scheme
pale_colors = {
    'W': "#560030", 'F': "#B50D6C", 'Y': "#CE4693",
    'A': '#E66101', 'V': "#A34A0A", 'L': "#7D3908", 'I': "#5B2804",
    'K': "#1978AF", 'R': "#032740", 'H': "#76BDE7",
    'D': '#7C131C', 'E': "#DA4760",
    'S': '#1A9850', 'T': "#26C46B", 'N': "#085719", 'Q': "#A7F0C7",
    'G': '#CCCCCC', 'P': '#CCCCCC', 'C': '#CCCCCC', 'M': '#CCCCCC'
}

# --- Plot figure (reuses `probs` from the previous cell)
# fig, ax = plt.subplots(figsize=(min(16, 0.6*L_mode + 2), 4), dpi=300)
fig, ax = plt.subplots(figsize=(6, 6))
lm.Logo(
    probs.fillna(0.0),
    ax=ax,
    shade_below=0.5,
    fade_below=0.5,
    font_name='Arial',
    color_scheme=pale_colors
)

# Axis labels and title
ax.set_ylabel("Probability", fontsize=44)
ax.set_xlabel("Position", fontsize=44)
ax.set_title("WebLogo – red peptides", fontsize=13, fontweight='bold', y=1.1)
ax.set_ylim(0, 1.0)
# Y-axis: ticks at 0, 0.5, 1
ax.yaxis.set_major_locator(FixedLocator([0, 0.5, 1.0]))
# X-axis: only integer ticks
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.tick_params(axis='both', which='major', labelsize=36)

# Journal-style aesthetics
ax.grid(False)
for spine in ax.spines.values():
    spine.set_linewidth(0.8)

# Layout & save
plt.tight_layout()
weblogo_png = base_dir / "weblogo_red_peptides_pale.png"
plt.savefig(weblogo_png, bbox_inches='tight', dpi=300)
plt.show()
print(f"[OK] Saved WebLogo → {weblogo_png}")

In [ ]:
# Heatmap of AA % by position (row-normalized, swapped axes)

# Use aligned sequences from the WebLogo step
if 'seqs_aligned' not in globals() or len(seqs_aligned) == 0:
    raise RuntimeError("Aligned sequences not found. Run the WebLogo cell first.")

# Amino acids to track
if 'amino_acids_to_track' in globals() and isinstance(amino_acids_to_track, (list, tuple)):
    aa_list = sorted(set([aa for aa in amino_acids_to_track if isinstance(aa, str) and len(aa) == 1]))
    if not aa_list:
        aa_list = list("ACDEFGHIKLMNPQRSTVWY")
else:
    aa_list = list("ACDEFGHIKLMNPQRSTVWY")

# Build frequency (%)
freq = pd.DataFrame(0.0, index=np.arange(1, L_mode + 1), columns=aa_list)
for s in seqs_aligned:
    for i, aa in enumerate(s, start=1):
        if aa in freq.columns:
            freq.at[i, aa] += 1.0

freq_pct = freq.div(len(seqs_aligned)).mul(100.0)

# Save raw frequency table
heatmap_csv = base_dir / "red_peptides_aa_frequency_percent.csv"
freq_pct.to_csv(heatmap_csv, index=True)
print(f"[OK] Saved frequency table → {heatmap_csv}")

# --- Normalize per row ---
row_norm = freq_pct.div(freq_pct.max(axis=1).replace(0, np.nan), axis=0)

# --- Transpose for swapped axes ---
row_norm_T = row_norm.T
freq_pct_T = freq_pct.T  # needed for annotations

# Plot row-normalized heatmap (AA on y-axis, position on x-axis)
fig, ax = plt.subplots(figsize=(max(8, 0.5 * L_mode + 2), max(4, 0.35 * len(aa_list) + 1)), dpi=150)

im = ax.imshow(row_norm_T.values, aspect='auto', cmap='viridis', vmin=0, vmax=1)

# Axes
ax.set_xticks(np.arange(L_mode))
ax.set_xticklabels(np.arange(1, L_mode + 1))
ax.set_yticks(np.arange(len(aa_list)))
ax.set_yticklabels(aa_list)
ax.set_xlabel("Position (1-based)")
ax.set_ylabel("Amino acid")
ax.set_title("AA frequency (row-normalized) by position – red peptides")
ax.grid(False)

# Colorbar
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Normalized frequency (per position)")

# Annotate with actual % (from freq_pct_T)
for i in range(freq_pct_T.shape[0]):      # amino acids
    for j in range(freq_pct_T.shape[1]):  # positions
        v = freq_pct_T.iat[i, j]
        if v >= 1.0:  # annotate only if >=1% for readability
            ax.text(j, i, f"{v:.0f}%", ha="center", va="center",
                    color="black", fontsize=18)

plt.tight_layout()
heatmap_png = base_dir / "heatmap_red_peptides_row_normalized_swapped.png"
plt.savefig(heatmap_png, bbox_inches='tight', dpi=300)
plt.show()
print(f"[OK] Saved swapped row-normalized heatmap → {heatmap_png}")

In [ ]:
# Heatmap of AA N-fold deviation – journal style
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.ticker import FormatStrFormatter

# Ensure aligned sequences exist
if 'seqs_aligned' not in globals() or len(seqs_aligned) == 0:
    raise RuntimeError("Aligned sequences not found. Run the WebLogo cell first.")

# Build N-fold deviation table
heatmap_data = pd.DataFrame(0.0, index=range(len(allowed_positions)), columns=amino_acids_to_track)

for pos in allowed_positions:
    if pos in specific_positions:
        aa_allowed = [specific_letters[specific_positions.index(pos)]]
    elif pos in allowed_positions:
        aa_allowed = list(allowed_letters)
    else:
        continue

    counts = {aa: 0 for aa in aa_allowed}
    for seq in seqs_aligned:
        if pos < len(seq):
            aa = seq[pos]
            if aa in counts:
                counts[aa] += 1

    total = sum(counts.values())
    for aa in aa_allowed:
        obs_freq = counts[aa] / total if total > 0 else 0
        expected_freq = 1.0 / len(aa_allowed)
        heatmap_data.at[pos, aa] = obs_freq / expected_freq if expected_freq > 0 else np.nan

# Remove columns that are all zeros
heatmap_data = heatmap_data.loc[:, (heatmap_data != 0).any(axis=0)]
row_norm_T = heatmap_data.T

# --- Create custom colormap ---
colors = ["#31669f", "#ffffff", "#a94545"]  # blue → white → red
cmap = LinearSegmentedColormap.from_list("pale_red_blue", colors)
norm = Normalize(vmin=0.5, vmax=2.0)

# --- Plot heatmap ---
fig, ax = plt.subplots(
    figsize=(max(8, 0.1 * length_criteria + 2), max(4, 0.25 * len(row_norm_T.index) + 1)),
    dpi=300
)

# Mask zero/NaN to show empty cells
masked_array = np.ma.masked_where(row_norm_T.values == 0, row_norm_T.values)
im = ax.imshow(masked_array, aspect='auto', cmap=cmap, norm=norm)

# Overlay cells with exactly 1x as light grey
for i in range(row_norm_T.shape[0]):
    for j in range(row_norm_T.shape[1]):
        v = row_norm_T.iat[i, j]
        if v == 1:
            rect = plt.Rectangle((j - 0.5, i - 0.5), 1, 1, facecolor="#d3d3d3", edgecolor="none")
            ax.add_patch(rect)

# Axes
ax.set_xticks(np.arange(len(row_norm_T.columns)))
ax.set_xticklabels([p + 1 for p in row_norm_T.columns], fontsize=18)
ax.set_yticks(np.arange(len(row_norm_T.index)))
ax.set_yticklabels(row_norm_T.index, fontsize=18)
ax.set_xlabel("position", fontsize=18)
ax.set_ylabel("amino acid", fontsize=18)
ax.grid(False)
# ax.set_title("AA N-fold deviation – red peptides", fontsize=18, pad=10)

# Journal-style ticks and spines
ax.tick_params(axis='both', which='major', direction='in', length=6, width=0.8,
               color='black', top=True, right=True)
ax.tick_params(axis='both', which='minor', bottom=False, top=False, left=False, right=False)
ax.minorticks_off()
for spine in ax.spines.values():
    spine.set_linewidth(0.8)

# Colorbar
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("N-fold deviation", fontsize=18)
cbar.ax.tick_params(labelsize=14)
cbar.ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))  # one decimal

# Annotate each cell with N-fold values
for i in range(row_norm_T.shape[0]):
    for j in range(row_norm_T.shape[1]):
        v = row_norm_T.iat[i, j]
        if not np.isnan(v) and v > 0:
            ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=12, color="black")

# Save
plt.tight_layout()
heatmap_png = base_dir / "heatmap_red_peptides_nfold_journal_style.png"
plt.savefig(heatmap_png, bbox_inches='tight', dpi=300)
plt.show()
print(f"[OK] Saved heatmap → {heatmap_png}")

In [ ]:
# Interactive volcano plot (Plotly)
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# 1) Renderer: 'vscode' works in VS Code with the Plotly extension; otherwise use 'browser'
pio.renderers.default = 'browser'

# 2) Category assignment without .apply
conds = [
    (df['p-value'] < 0.2) & (df['Rel_Enrichment'] > 2),
    (df['p-value'] < 0.2) & (df['Rel_Enrichment'] < 0.5)
]
choices = ['Up', 'Down']
df['Category'] = np.select(conds, choices, default='NS')

# 3) Interactive scatter (WebGL)
fig = px.scatter(
    df,
    x='Log2_Enrichment',
    y='Neg_Log10_P',
    color='Category',
    hover_name='Peptide',
    hover_data={'p-value': ':.3g', 'Rel_Enrichment': ':.3g'},
    color_discrete_map={'Up': 'red', 'Down': 'blue', 'NS': 'grey'},
    title='Volcano Plot',
    render_mode='webgl'     # big speed win for large N
)

# 4) Threshold lines
fig.add_hline(y=-np.log10(0.2), line_dash="dash", line_color="black")
fig.add_vline(x=np.log2(2),     line_dash="dash", line_color="black")
fig.add_vline(x=np.log2(0.5),   line_dash="dash", line_color="black")

# 5) Text labels for the top-N most significant "Up" points
TOP_N = 50
mask_up = (df['Category'] == 'Up')
top_up = df.loc[mask_up].nlargest(TOP_N, 'Neg_Log10_P')

fig.add_trace(go.Scatter(
    x=top_up['Log2_Enrichment'],
    y=top_up['Neg_Log10_P'],
    mode='text',
    text=top_up['Peptide'],
    textposition='top center',
    showlegend=False,
    hoverinfo='skip'  # labels already visible; hover handled by main trace
))

# 6) Display + write HTML
fig.show()
fig.write_html(base_dir / 'volcano_plot.html', auto_open=False)
print("Saved interactive plot to volcano_plot.html")